# 03 Romberg 求积与自适应 Simpson

复合公式通过减小步长降低误差。Romberg 求积进一步利用误差展开，把不同步长的梯形公式组合起来消去主误差项。自适应 Simpson 则根据局部误差估计决定在哪里细分区间。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import adaptive_simpson, composite_trapezoid, romberg_integrate


## Richardson 外推

假设某个近似满足

$$
A(h)=I+c_1h^p+c_2h^{p+q}+\cdots.
$$

用步长 $h$ 和 $h/2$ 的两个近似，可以消去主误差项：

$$
I\approx \frac{2^pA(h/2)-A(h)}{2^p-1}.
$$

复合梯形公式对光滑函数具有偶次误差展开：

$$
T(h)=I+c_1h^2+c_2h^4+c_3h^6+\cdots.
$$

因此 Romberg 表递推为

$$
R_{k,j}=R_{k,j-1}+\frac{R_{k,j-1}-R_{k-1,j-1}}{4^j-1}.
$$


In [ ]:
def teaching_romberg(func, a, b, levels):
    table = np.full((levels, levels), np.nan)
    for k in range(levels):
        n = 2**k
        table[k, 0] = composite_trapezoid(func, a, b, n)
        for j in range(1, k + 1):
            table[k, j] = table[k, j - 1] + (table[k, j - 1] - table[k - 1, j - 1]) / (4**j - 1)
    return table

romberg_table = teaching_romberg(np.sin, 0.0, math.pi, levels=6)
print(np.array2string(romberg_table, precision=10, suppress_small=False))
print("对角线误差:", np.abs(np.diag(romberg_table) - 2.0))


## Romberg 算法设计

直接重新计算每一层复合梯形会浪费函数值。更常用的递推是：从 $2^{k-1}$ 个小区间变成 $2^k$ 个小区间时，只需要计算上一层没有出现过的中点。

终止准则可以使用相邻对角线差值：

$$
|R_{k,k}-R_{k-1,k-1}|<\varepsilon.
$$

这个准则不是严格误差保证，但对光滑函数通常很有效。


In [ ]:
formal = romberg_integrate(math.sin, 0.0, math.pi, max_order=8, tolerance=1e-12)
print("Romberg 正式实现结果:", formal.value)
print("是否达到终止准则:", formal.converged)
print("迭代层数:", formal.iterations)


## 自适应 Simpson 的局部误差估计

对区间 $[a,b]$，记单区间 Simpson 为 $S(a,b)$。二分后得到

$$
S(a,m)+S(m,b),\qquad m=\frac{a+b}{2}.
$$

Simpson 误差主项可推出局部误差估计

$$
\epsilon\approx \frac{|S(a,m)+S(m,b)-S(a,b)|}{15}.
$$

如果误差小于局部容差，就接受区间；否则继续二分。实现时应复用端点和中点函数值，并设置最大递归深度。


In [ ]:
def teaching_adaptive_simpson(func, a, b, tol, max_depth=16):
    intervals = []

    def simpson_from_values(left, right, f_left, f_mid, f_right):
        return (right - left) * (f_left + 4.0 * f_mid + f_right) / 6.0

    def refine(left, right, f_left, f_mid, f_right, whole, local_tol, depth):
        mid = 0.5 * (left + right)
        left_mid = 0.5 * (left + mid)
        right_mid = 0.5 * (mid + right)
        f_left_mid = func(left_mid)
        f_right_mid = func(right_mid)
        left_value = simpson_from_values(left, mid, f_left, f_left_mid, f_mid)
        right_value = simpson_from_values(mid, right, f_mid, f_right_mid, f_right)
        refined = left_value + right_value
        error = abs(refined - whole) / 15.0
        if error <= local_tol or depth >= max_depth:
            intervals.append((left, right, error, depth))
            return refined + (refined - whole) / 15.0
        return refine(left, mid, f_left, f_left_mid, f_mid, left_value, local_tol / 2.0, depth + 1) + refine(
            mid, right, f_mid, f_right_mid, f_right, right_value, local_tol / 2.0, depth + 1
        )

    mid = 0.5 * (a + b)
    f_a, f_mid, f_b = func(a), func(mid), func(b)
    initial = simpson_from_values(a, b, f_a, f_mid, f_b)
    value = refine(a, b, f_a, f_mid, f_b, initial, tol, 0)
    return value, np.array(intervals)

peaked = lambda x: 1.0 / (1.0 + 300.0 * (x - 0.35) ** 2)
value, intervals = teaching_adaptive_simpson(peaked, 0.0, 1.0, tol=1e-7, max_depth=20)
print("教学版结果:", value)
print("接受区间数量:", intervals.shape[0])
print("最小区间长度:", np.min(intervals[:, 1] - intervals[:, 0]))


## 区间划分可视化

自适应方法的价值不只在最终数值，也在它给出的区间分布。对尖峰函数，算法应该在尖峰附近使用更短的区间。


In [ ]:
formal_adaptive = adaptive_simpson(peaked, 0.0, 1.0, tolerance=1e-7, max_depth=20)
x_plot = np.linspace(0.0, 1.0, 800)
y_plot = np.array([peaked(x) for x in x_plot])

plt.figure(figsize=(8, 4.4))
plt.plot(x_plot, y_plot, label="被积函数")
for left, right in formal_adaptive.intervals:
    plt.axvspan(left, right, color="C1", alpha=0.08)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("自适应 Simpson 接受区间：尖峰附近划分更密")
plt.legend()
plt.show()

print("正式实现结果:", formal_adaptive.value)
print("误差估计:", formal_adaptive.error_estimate)
print("函数计算次数:", formal_adaptive.evaluations)
print("是否未触及最大深度:", formal_adaptive.converged)


## 适用条件、失败情形和小结

* Romberg 依赖光滑函数的偶次误差展开。函数不光滑、端点奇异或含噪声时，外推可能失效。
* 自适应 Simpson 适合局部尖峰、边界层或变化不均匀的函数，但误差估计仍建立在局部光滑假设上。
* 自适应递归必须有最大深度和未收敛报告，否则可能在奇异点附近无限细分。
* 两类方法都不是“免费提高精度”：Romberg 使用外推假设，自适应方法增加控制逻辑和函数值管理。
